# Loop 03 — building the blind re-rating sheet

This notebook builds the **30-row blind re-rating sheet** designed in this loop's
[README](README.md): the 4 feature-nominated `not` rows embedded among 26 decoys
drawn uniformly from the remaining `not` rows, shuffled so the nominees don't
cluster, with **no citation counts anywhere**.

Inputs (both committed by earlier loops, no live API or corpus access needed):

- loop 01's `labeling-sheet.csv` — row metadata (title, authors, year, arXiv id)
  for all 150 sampled anchor rows;
- loop 02's `features.csv` — the anchor labels plus the max-endorsement feature
  that nominated the four rows.

Outputs, written beside this notebook:

- `rating-sheet.csv` — the blind sheet raters see: metadata columns plus empty
  `label`/`notes`. One copy per rater, labels come back as per-rater columns.
- `sheet-key.csv` — the analysis key (which rows are nominees, their seeds and
  original provenance). **Never shown to raters.**

In [1]:
"""Build loop 03's blind re-rating sheet from loops 01 and 02's committed data."""

import hashlib
from pathlib import Path

import pandas

WORKSTREAM_DIR = Path.cwd().parent if Path.cwd().name == "03_targeted-rerating" else Path.cwd()
LOOP_DIR = WORKSTREAM_DIR / "03_targeted-rerating"

# Loop 01's sheet carries the row metadata; loop 02's features carry the labels
# and the endorsement feature that nominated the four boundary rows.
metadata = pandas.read_csv(WORKSTREAM_DIR / "01_hand-labeled-anchor" / "labeling-sheet.csv")
features = pandas.read_csv(WORKSTREAM_DIR / "02_downstream-endorsement" / "features.csv")

not_rows = features[features["label"] == "not"]
assert len(not_rows) == 135, f"expected the anchor's 135 nots, got {len(not_rows)}"
print(f"{len(metadata)} metadata rows, {len(not_rows)} labeled nots")

150 metadata rows, 135 labeled nots


## The four nominees

Loop 02's strip plot put four `not` rows above the ~10³ max-endorsement line the
researcher eyeballed (2026-07-25): the 2,220-citer multiagent-RL survey and three
one-big-citer artifacts (DAMO-YOLO, *Optimistic Policy Iteration*, TransOMCS).
The feature only *nominates* — per the circularity guard, relabeling is reserved
for the blind raters this sheet is for. The cell below re-derives the four from
the committed feature table and asserts they are exactly the expected rows.

In [2]:
# The ~10^3 line separates the four nominees from the next-highest not (max
# endorsement 6,003 / 4,820 / 1,882 / 1,552 vs. a sub-1,000 field below).
ENDORSEMENT_LINE = 1_000

# Corpusids pinned in this loop's README (provenance: loop 02's results).
EXPECTED_NOMINEES = {
    206794869,  # A Comprehensive Survey of Multiagent Reinforcement Learning (2008)
    254043744,  # DAMO-YOLO (2022)
    1472007,    # On the Convergence of Optimistic Policy Iteration (2002)
    218470275,  # TransOMCS (2020)
}

nominee_rows = not_rows[not_rows["max_endorsement"] >= ENDORSEMENT_LINE]
assert set(nominee_rows["citer_corpusid"]) == EXPECTED_NOMINEES, "nominee set drifted"

decoy_pool = not_rows[not_rows["max_endorsement"] < ENDORSEMENT_LINE]
assert len(decoy_pool) == 131, f"expected 131 remaining nots, got {len(decoy_pool)}"
print(nominee_rows[["citer_corpusid", "downstream_citers", "max_endorsement"]].to_string(index=False))

 citer_corpusid  downstream_citers  max_endorsement
      218470275                 98             1552
      206794869               2220             6003
        1472007                 97             1882
      254043744                337             4820


## Drawing the 26 decoys — loop 01's id-hash order, over distinct papers

Same draw as loop 01's blind sample: order candidates by the MD5 hash of the
citer's corpusid and take the first 26. Deterministic (the sheet is reproducible
run-to-run) and independent of every paper property — a hash of an arbitrary id
knows nothing about citations, age, or field, so the draw stays metric-blind.

**Data note found while building the draw:** the anchor's 135 `not` rows are
only **123 distinct papers** — ten citers were sampled independently under 2–3
different seeds (loop 01 sampled per seed, and big-pool citers overlap). Loops
01–02 operated on rows, so this never mattered; a rating sheet must show
distinct papers, so the draw here runs over **unique citer ids** (the design's
"131 remaining `not` rows" are 119 distinct candidate papers — the 4 nominees
are duplicated by no seed). This also evens the draw: under row-level ordering a
twice-sampled paper would have entered the sort twice.

In [3]:
DECOY_COUNT = 26


def id_hash(corpusid: int, salt: str = "") -> str:
    """MD5 hex digest of a corpusid, optionally salted.

    Args:
        corpusid: The id to hash.
        salt: Prefix mixed into the hash, giving an independent ordering
            (empty for loop 01's draw order).

    Returns:
        The hex digest ordering key.
    """
    return hashlib.md5((salt + str(corpusid)).encode()).hexdigest()


candidate_ids = set(decoy_pool["citer_corpusid"])
assert not candidate_ids & EXPECTED_NOMINEES, "nominees must not re-enter as decoys"
assert len(candidate_ids) == 119, f"expected 119 distinct candidates, got {len(candidate_ids)}"

decoy_ids = sorted(candidate_ids, key=id_hash)[:DECOY_COUNT]
sheet_ids = set(decoy_ids) | EXPECTED_NOMINEES
assert len(sheet_ids) == 30, f"expected 30 sheet rows, got {len(sheet_ids)}"
print(f"{len(decoy_ids)} decoys drawn from {len(candidate_ids)} distinct candidates "
      f"({len(decoy_pool)} rows)")

26 decoys drawn from 119 distinct candidates (131 rows)


## Presentation order — the same hash would cluster the nominees

The README says to shuffle presentation rows "by the same hash". Taken literally
that backfires: the decoys *are* the 26 smallest hashes among the 131 candidates,
so they occupy roughly the bottom fifth of hash space — while the four nominees'
hashes are unconstrained. Sorting the combined 30 by the draw hash therefore
tends to sink every nominee below every decoy (each nominee lands above the decoy
band with probability ≈ 1 − 26/131 ≈ 0.80), which is exactly the clustering the
shuffle is meant to prevent.

Fix, keeping the design's actual intent (deterministic, metric-blind, no
clustering): order the sheet by a **salted** variant of the same hash
(`md5("order:" + corpusid)`), which is independent of the draw ordering. The
cell below shows the nominee positions under both orderings.

In [4]:
def nominee_positions(ordering_salt: str) -> list[int]:
    """1-based sheet positions of the four nominees under a hash ordering.

    Args:
        ordering_salt: Salt passed to ``id_hash`` for the ordering.

    Returns:
        Sorted positions (1–30) the nominees would occupy.
    """
    ordered = sorted(sheet_ids, key=lambda corpusid: id_hash(corpusid, ordering_salt))
    return sorted(ordered.index(corpusid) + 1 for corpusid in EXPECTED_NOMINEES)


print(f"draw-hash order (salt=''):    nominees at {nominee_positions('')}")
print(f"salted order (salt='order:'): nominees at {nominee_positions('order:')}")

# The salted ordering is the one the sheet ships with.
sheet_order = sorted(sheet_ids, key=lambda corpusid: id_hash(corpusid, "order:"))

draw-hash order (salt=''):    nominees at [27, 28, 29, 30]
salted order (salt='order:'): nominees at [1, 7, 13, 24]


In [5]:
# Assemble the sheet: metadata joined by corpusid, presented in salted-hash
# order, label/notes blanked. No count or endorsement column comes anywhere
# near it, and neither does the seed (a grouping cue the raters don't need).
# A twice-sampled citer has one metadata row per seed — identical paper facts —
# so the sheet keeps one row per paper and the key aggregates the seeds.
sheet_metadata = (
    metadata[metadata["citer_corpusid"].isin(sheet_ids)]
    .drop_duplicates(subset="citer_corpusid")
    .set_index("citer_corpusid")
    .loc[sheet_order]
    .reset_index()
)
assert len(sheet_metadata) == 30, "every sheet row must appear in loop 01's metadata"

rating_sheet = sheet_metadata[["citer_corpusid", "arxiv_id", "year", "title", "authors"]].assign(
    label="", notes=""
)
rating_sheet["year"] = rating_sheet["year"].astype("Int64")

# The analysis key: nominee flag + the provenance the blind sheet withholds
# (seeds joined with ";" where a paper was sampled under more than one).
seeds_by_citer = metadata.groupby("citer_corpusid")["seed"].agg("; ".join)
sheet_key = sheet_metadata[["citer_corpusid", "title"]].assign(
    seeds=sheet_metadata["citer_corpusid"].map(seeds_by_citer),
    nominee=sheet_metadata["citer_corpusid"].isin(EXPECTED_NOMINEES),
)

rating_sheet.to_csv(LOOP_DIR / "rating-sheet.csv", index=False)
sheet_key.to_csv(LOOP_DIR / "sheet-key.csv", index=False)
print(f"rating-sheet.csv: {len(rating_sheet)} rows, columns {list(rating_sheet.columns)}")
rating_sheet.head(10)

rating-sheet.csv: 30 rows, columns ['citer_corpusid', 'arxiv_id', 'year', 'title', 'authors', 'label', 'notes']


,citer_corpusid,arxiv_id,year,title,authors,label,notes
0,254043744,2211.15444,2022,DAMO-YOLO : A Report on Real-Time Object Detec...,"Xianzhe Xu, Yiqi Jiang, Weihua Chen, Yi-Li Hua...",,
1,267312132,2401.15282,2024,GEM: Boost Simple Network for Glass Surface Se...,"Jing Hao, Moyun Liu, Kuo Feng Hung",,
2,277856775,2504.12817,2025,Explainable Scene Understanding with Qualitati...,"Nassim Belmecheri, Arnaud Gotlieb, Nadjib Laza...",,
3,278193103,NaN,2025,"Advancements in maize leaf disease detection, ...","Suresh Timilsina, Sandhya Sharma, Satoshi Kondo",,
4,266236087,NaN,2023,Robotic Grasping Based on Deep Learning: A Survey,"M. Rashed, R. N. Farhan, Wesam M. Jasim",,
5,283847587,NaN,2026,Pixel-scale satellite forecasting of cyanobact...,"M. Beal, B. Schaeffer",,
6,218470275,2005.00206,2020,TransOMCS: From Linguistic Graphs to Commonsen...,"Hongming Zhang, Daniel Khashabi, Yangqiu Song,...",,
7,251480548,NaN,2022,FH-YOLOv4 with Constrained Aspect Ratio Loss f...,"Yue Wang, L. Hong, Dewen Gu, P. Fu",,
8,282419912,NaN,2025,Lightweight and Improved PCBA Welding Defect D...,"Zhen Xu, Qingfeng Ma",,
9,278012917,NaN,2025,Decision support for the identification of tes...,"Serge Zaugg, Camille Vögeli, Lena Märki, C. Du...",,


## What happens next

The sheet above is the deliverable; the loop's experiment now leaves the repo:

1. Each of the 2–3 recruited raters gets **their own copy** of
   `rating-sheet.csv` with the empty `label` column, told nothing about how the
   rows were chosen — to them it's "which of these do you recognize as papers
   that matter?" under loop 01's blind rules (recognition only;
   `landmark` / `not` / `unsure`; no Google, no Semantic Scholar; arXiv abstract
   pages allowed; no guessing from title plausibility).
2. Labels come back as separate per-rater columns, never merged by discussion.
3. Analysis (a follow-up section here when sheets return): relabel rates among
   nominees vs. decoys, per-rater and pooled, read against the outcome table in
   this loop's README. `sheet-key.csv` joins the returned labels back to
   nominee/decoy status.